# 🏆 Master Benchmark Comparison & Publication Visualizations
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization

---
## 📌 Strict Scientific Benchmark Setup
- **Identical Token Limits Across All Models**: `MAX_SOURCE_LENGTH = 512` & `MAX_TARGET_LENGTH = 128`
- **The 3 Non-Redundant Core Research Pillars Evaluated:**

| Pillar | Core Metric | What It Means | Direction (Better) |
| :--- | :--- | :--- | :---: |
| **Pillar 1: Overall Quality** | **ROUGE-L F1** & **Target F1** | Syntactic fluency and semantic overlap with ground-truth author highlights | **Higher is Better (↑)** |
| **Pillar 2: Hallucination Risk** | **Ungrounded Token %** & **Source Grounding %** | Percentage of claims made that have ZERO basis in the source paper abstract | **Lower is Better (↓)** |
| **Pillar 3: Inference Time** | **Wall-Clock Latency (sec)** & **Throughput (tok/s)** | Real-world GPU execution time per paper and generation serving speed | **Lower Latency (↓) / Higher Tok/s (↑)** |


In [ ]:
# ================================================================
# MASTER BENCHMARK COMPARISON & PUBLICATION VISUALIZATIONS
# FIRE MixSub-SciHigh | Jadavpur University
# One-shot, schema-compatible, defensive implementation
# ================================================================

# 1. Install/check dependencies
import sys
import subprocess
import importlib.util

for package, module in [
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
RESULTS_DIR = os.path.join(BASE_DIR, "results")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Results folder:", RESULTS_DIR)

# 3. Find benchmark JSON files
json_files = sorted(
    glob.glob(os.path.join(RESULTS_DIR, "benchmark_*.json"))
)

if not json_files:
    raise FileNotFoundError(
        "No benchmark_*.json files found in results folder. "
        "Run the individual model benchmark notebooks first."
    )

print(f"\nFound {len(json_files)} benchmark JSON file(s):")
for path in json_files:
    print(" -", os.path.basename(path))


# 4. Metric extraction helpers
def first_available(data, keys, default=np.nan):
    """Return the first non-null value found among candidate keys."""
    for key in keys:
        value = data.get(key)
        if value is not None:
            try:
                value = float(value)
                if np.isfinite(value):
                    return value
            except (TypeError, ValueError):
                continue
    return default


def normalize_fraction_or_percent(value):
    """
    Convert fractions to percentages.
    Values already expressed as percentages remain unchanged.
    """
    if pd.isna(value):
        return np.nan
    return value * 100 if 0 <= value <= 1 else value


def load_benchmark(path):
    """Read both nested and flat benchmark JSON formats."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError("Benchmark JSON root must be an object.")

    # Newer notebooks store metrics inside the 'metrics' object.
    metrics = raw.get("metrics", raw)

    if not isinstance(metrics, dict):
        metrics = raw

    model_name = raw.get(
        "model",
        raw.get("model_name", os.path.basename(path)
                .replace("benchmark_", "")
                .replace(".json", ""))
    )

    quality = first_available(metrics, [
        "rougeL_f1",
        "overall_quality_rougeL_f1",
        "rouge_l_f1",
        "rougeL_f1_score"
    ])

    target_f1 = first_available(metrics, [
        "target_lexical_f1",
        "f1_score_target",
        "target_f1",
        "target_f1_score"
    ])

    source_precision = first_available(metrics, [
        "source_lexical_precision",
        "source_grounding_precision",
        "source_precision"
    ])

    risk = first_available(metrics, [
        "hallucination_risk_pct",
        "ungrounded_token_pct",
        "hallucination_risk"
    ])

    # Derive risk from source overlap only when risk is absent.
    if pd.isna(risk) and not pd.isna(source_precision):
        risk = 1 - source_precision

    risk = normalize_fraction_or_percent(risk)

    # Older notebooks may store source grounding as a fraction.
    if not pd.isna(source_precision):
        source_precision = normalize_fraction_or_percent(
            source_precision
        )

    latency = first_available(metrics, [
        "mean_latency_seconds",
        "wall_clock_latency_sec",
        "mean_latency_sec",
        "average_latency_seconds"
    ])

    throughput_samples = first_available(metrics, [
        "throughput_samples_per_second",
        "samples_per_second",
        "throughput_samples_sec"
    ])

    throughput_tokens = first_available(metrics, [
        "throughput_tokens_per_sec",
        "tokens_per_second",
        "generation_tokens_per_second"
    ])

    # Support alternate legacy latency naming.
    if pd.isna(latency):
        total_latency = first_available(metrics, [
            "total_latency_seconds",
            "total_latency_sec"
        ])
        sample_count = first_available(metrics, [
            "samples",
            "num_samples",
            "evaluation_samples"
        ])

        if not pd.isna(total_latency) and not pd.isna(sample_count):
            if sample_count > 0:
                latency = total_latency / sample_count

    # Derive sample throughput from mean latency if necessary.
    if pd.isna(throughput_samples) and not pd.isna(latency):
        if latency > 0:
            throughput_samples = 1 / latency

    return {
        "Model": str(model_name),
        "ROUGE-L F1 (%)": quality * 100 if not pd.isna(quality) else np.nan,
        "Target Lexical F1 (%)": target_f1 * 100 if not pd.isna(target_f1) else np.nan,
        "Source Lexical Precision (%)": source_precision,
        "Ungrounded-token Proxy (%)": risk,
        "Mean Latency (seconds/sample)": latency,
        "Throughput (samples/second)": throughput_samples,
        "Throughput (tokens/second)": throughput_tokens,
        "Evaluation Type": raw.get(
            "evaluation_type",
            "Not specified"
        ),
        "Fine-tuning": raw.get(
            "fine_tuning",
            "Not specified"
        ),
        "Source File": os.path.basename(path)
    }


# 5. Load all benchmark files safely
records = []
failed_files = []

for path in json_files:
    try:
        record = load_benchmark(path)
        records.append(record)
    except Exception as e:
        failed_files.append((os.path.basename(path), str(e)))
        print(f"WARNING: Could not read {os.path.basename(path)}: {e}")

if not records:
    raise ValueError(
        "No valid benchmark JSON files could be loaded."
    )

df = pd.DataFrame(records)

# Remove duplicate model entries, keeping the latest sorted filename.
df = df.drop_duplicates(subset=["Model"], keep="last")
df = df.sort_values("Model").reset_index(drop=True)

# 6. Save complete master comparison CSV
master_csv = os.path.join(
    RESULTS_DIR,
    "master_3pillar_benchmark_comparison.csv"
)

df.to_csv(master_csv, index=False)

print("\n" + "=" * 100)
print("MASTER THREE-PILLAR BENCHMARK COMPARISON")
print("=" * 100)

# Bypassing Colab's rich display format_name bug by printing the DataFrame directly
print(df.round(4).to_string(index=False))

print("\nMaster CSV saved:", master_csv)


# 7. Publication plot configuration
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False
})

models = df["Model"].astype(str).tolist()
x_positions = np.arange(len(models))

def save_figure(fig, filename):
    path = os.path.join(PLOTS_DIR, filename)
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()
    plt.close(fig)
    print("Saved:", path)


# 8. Figure 1 — Quality vs inference latency
plot_df = df.dropna(
    subset=["ROUGE-L F1 (%)", "Mean Latency (seconds/sample)"]
)

if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(
        plot_df["Mean Latency (seconds/sample)"],
        plot_df["ROUGE-L F1 (%)"],
        s=140,
        edgecolors="black",
        linewidths=0.8,
        alpha=0.85
    )

    for _, row in plot_df.iterrows():
        ax.annotate(
            row["Model"],
            (
                row["Mean Latency (seconds/sample)"],
                row["ROUGE-L F1 (%)"]
            ),
            xytext=(7, 6),
            textcoords="offset points",
            fontsize=9
        )

    ax.set_title(
        "Pillar 1 vs Pillar 3: Quality vs Inference Latency"
    )
    ax.set_xlabel("Mean Latency (seconds/sample) — Lower is Better")
    ax.set_ylabel("ROUGE-L F1 (%) — Higher is Better")
    ax.grid(alpha=0.25)

    save_figure(fig, "quality_vs_inference_latency.png")
else:
    print("Skipping Figure 1: required metrics unavailable.")


# 9. Figure 2 — Quality vs grounding-risk proxy
plot_df = df.dropna(
    subset=["ROUGE-L F1 (%)", "Ungrounded-token Proxy (%)"]
)

if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(
        plot_df["Ungrounded-token Proxy (%)"],
        plot_df["ROUGE-L F1 (%)"],
        s=140,
        edgecolors="black",
        linewidths=0.8,
        alpha=0.85
    )

    for _, row in plot_df.iterrows():
        ax.annotate(
            row["Model"],
            (
                row["Ungrounded-token Proxy (%)"],
                row["ROUGE-L F1 (%)"]
            ),
            xytext=(7, 6),
            textcoords="offset points",
            fontsize=9
        )

    ax.set_title(
        "Pillar 1 vs Pillar 2: Quality vs Ungrounded-token Proxy"
    )
    ax.set_xlabel(
        "Ungrounded-token Proxy (%) — Lower is Better"
    )
    ax.set_ylabel("ROUGE-L F1 (%) — Higher is Better")
    ax.grid(alpha=0.25)

    save_figure(fig, "quality_vs_hallucination_risk.png")
else:
    print("Skipping Figure 2: required metrics unavailable.")


# 10. Figure 3 — Grounding-risk proxy comparison
plot_df = df.dropna(subset=["Ungrounded-token Proxy (%)"])

if not plot_df.empty:
    plot_df = plot_df.sort_values("Ungrounded-token Proxy (%)")

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.bar(
        plot_df["Model"],
        plot_df["Ungrounded-token Proxy (%)"],
        edgecolor="black",
        linewidth=0.6
    )

    ax.axhline(
        plot_df["Ungrounded-token Proxy (%)"].mean(),
        linestyle="--",
        linewidth=1.5,
        label="Model average"
    )

    ax.set_title(
        "Pillar 2: Source-grounding Risk Proxy Across Models"
    )
    ax.set_ylabel("Ungrounded-token Proxy (%) — Lower is Better")
    ax.set_xlabel("Model")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    save_figure(fig, "hallucination_risk_ranking.png")
else:
    print("Skipping Figure 3: risk metric unavailable.")


# 11. Figure 4 — Inference speed
latency_df = df.dropna(
    subset=["Mean Latency (seconds/sample)"]
)

throughput_df = df.dropna(
    subset=["Throughput (samples/second)"]
)

if not latency_df.empty or not throughput_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    if not latency_df.empty:
        axes[0].bar(
            latency_df["Model"],
            latency_df["Mean Latency (seconds/sample)"],
            edgecolor="black",
            linewidth=0.6
        )
        axes[0].set_title("Mean Inference Latency")
        axes[0].set_ylabel("Seconds per Sample — Lower is Better")
        axes[0].tick_params(axis="x", rotation=25)
        axes[0].grid(axis="y", alpha=0.25)
    else:
        axes[0].text(
            0.5, 0.5, "Latency data unavailable",
            ha="center", va="center"
        )
        axes[0].set_axis_off()

    if not throughput_df.empty:
        axes[1].bar(
            throughput_df["Model"],
            throughput_df["Throughput (samples/second)"],
            edgecolor="black",
            linewidth=0.6
        )
        axes[1].set_title("Inference Throughput")
        axes[1].set_ylabel("Samples per Second — Higher is Better")
        axes[1].tick_params(axis="x", rotation=25)
        axes[1].grid(axis="y", alpha=0.25)
    else:
        axes[1].text(
            0.5, 0.5, "Throughput data unavailable",
            ha="center", va="center"
        )
        axes[1].set_axis_off()

    fig.suptitle("Pillar 3: Inference Speed Comparison", y=1.03)
    plt.tight_layout()

    save_figure(fig, "inference_speed_latency.png")
else:
    print("Skipping Figure 4: speed metrics unavailable.")


# 12. Save a compact run report
report = {
    "benchmark_files_found": len(json_files),
    "valid_models_loaded": len(df),
    "failed_files": failed_files,
    "models": models,
    "comparison_csv": master_csv,
    "plots_directory": PLOTS_DIR,
    "generated_plots": sorted(
        os.path.basename(p)
        for p in glob.glob(os.path.join(PLOTS_DIR, "*.png"))
    )
}

report_path = os.path.join(
    RESULTS_DIR,
    "master_benchmark_run_report.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("\n" + "=" * 70)
print("BENCHMARK COMPARISON COMPLETED")
print("=" * 70)
print("Valid models:", len(df))
print("CSV:", master_csv)
print("Plots:", PLOTS_DIR)
print("Run report:", report_path)

if failed_files:
    print("\nSome files could not be loaded:")
    for filename, error in failed_files:
        print("-", filename, ":", error)

print("\nAll available outputs have been saved to Google Drive.")